# Maharashtra Saree HMR 2.0 | Kaggle notebook

This notebook evaluates official 4D-Humans/HMR2 predictions on the saree image dataset. It saves image inventory, rendered PNGs, OBJ meshes, per-image status, failures and experiment metadata.

**Setup:** Attach a private Kaggle Dataset containing the cleaned asset ZIP or original full bundle. Enable Internet and select an available GPU. Run cells in order. This notebook uses Kaggle's active Python interpreter and does not reinstall torch/torchvision.

## 1. Environment validation

In [ ]:
import os, sys, json, time, shutil, hashlib, zipfile, platform, subprocess, random, traceback, runpy
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, cv2

import torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU not active. Kaggle Settings > Accelerator > GPU, then restart session.")
print("GPU:", torch.cuda.get_device_name(0))


## 2. Experiment paths

In [ ]:
WORK = Path("/kaggle/working")
ROOT = WORK / "Saree_HMR_Research"
DIRS = {name: ROOT / name for name in [
 "01_Dataset_Inventory","02_Environment_Logs","03_HMR_Outputs",
 "04_Rendered_Meshes","05_Failed_Images","06_Experiment_Metadata","07_Evaluation"]}
for p in DIRS.values(): p.mkdir(parents=True, exist_ok=True)
REPO = WORK / "4D-Humans"
SMPL_NAME = "basicModel_neutral_lbs_10_207_0_v1.0.0.pkl"
DATASET_NAME = "MAHARASTRA-20261102T145611Z-1-001.zip"
EXTS = {".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff"}
ASSET_DIR = WORK / "assets"; ASSET_DIR.mkdir(exist_ok=True)
DATA_ZIP = ASSET_DIR / "dataset_images.zip"
SMPL = ASSET_DIR / SMPL_NAME
def sha256(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()


## 3. Robust asset discovery

Supports the cleaned ZIP with images + SMPL inside, the original full bundle ZIP with nested dataset ZIP + SMPL, separate original files, and Kaggle-extracted input folders. It prints what Kaggle actually mounted.

In [ ]:
INPUT = Path("/kaggle/input")
if not INPUT.exists() or not any(INPUT.iterdir()):
    raise RuntimeError("No Kaggle input attached. Click Add Input and attach your private dataset.")
files = [p for p in INPUT.rglob("*") if p.is_file()]
print("Input files:", len(files))
for p in files[:80]: print(p.relative_to(INPUT))
asset_source = None
datasets = [p for p in files if p.name == DATASET_NAME]
smpls = [p for p in files if p.name == SMPL_NAME]
if len(datasets)==1 and len(smpls)==1:
    shutil.copy2(datasets[0],DATA_ZIP); shutil.copy2(smpls[0],SMPL); asset_source="separate original files"

if asset_source is None:
    for zp in [p for p in files if p.suffix.lower()==".zip"]:
        try:
            with zipfile.ZipFile(zp) as z:
                names=[n for n in z.namelist() if not n.endswith("/")]
                sm=[n for n in names if Path(n).name==SMPL_NAME]
                imgs=[n for n in names if Path(n).suffix.lower() in EXTS]
                nested=[n for n in names if Path(n).name==DATASET_NAME]
                if len(sm)==1 and imgs:
                    with zipfile.ZipFile(DATA_ZIP,"w",zipfile.ZIP_DEFLATED) as out:
                        for n in imgs: out.writestr(n,z.read(n))
                    SMPL.write_bytes(z.read(sm[0])); asset_source=f"clean archive: {zp.name}"; break
                if len(sm)==1 and len(nested)==1:
                    DATA_ZIP.write_bytes(z.read(nested[0])); SMPL.write_bytes(z.read(sm[0]))
                    asset_source=f"original bundle: {zp.name}"; break
        except zipfile.BadZipFile: pass

if asset_source is None and len(smpls)==1:
    imgs=[p for p in files if p.suffix.lower() in EXTS]
    if imgs:
        shutil.copy2(smpls[0],SMPL)
        with zipfile.ZipFile(DATA_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
            for p in imgs: z.write(p,p.name)
        asset_source="Kaggle extracted folders"

if asset_source is None:
    raise RuntimeError("Assets not found. ZIP files: "+str([str(p) for p in files if p.suffix.lower()==".zip"])+"; PKL files: "+str([str(p) for p in files if p.suffix.lower()==".pkl"]))
if SMPL.stat().st_size < 1_000_000: raise RuntimeError("SMPL file appears incomplete.")
with zipfile.ZipFile(DATA_ZIP) as z:
    members=[n for n in z.namelist() if Path(n).suffix.lower() in EXTS and not n.endswith("/")]
if not members: raise RuntimeError("No supported images found in dataset archive.")
print("Asset source:",asset_source,"Images:",len(members),"SMPL bytes:",SMPL.stat().st_size)
print("SMPL SHA256:",sha256(SMPL))


## 4. Extract images and build inventory

Unreadable files are explicitly inventoried; duplicates are flagged by content hash rather than silently discarded.

In [ ]:
IMAGE_ROOT=WORK/"dataset_images"; IMAGE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as z: z.extractall(IMAGE_ROOT)
paths=sorted(p for p in IMAGE_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in EXTS)
if not paths: raise RuntimeError("Extraction produced no images.")
seen={}; rows=[]
for p in paths:
    h=sha256(p); im=cv2.imread(str(p))
    rows.append({"relative_path":str(p.relative_to(IMAGE_ROOT)),"bytes":p.stat().st_size,
                 "sha256":h,"duplicate_of":seen.get(h),"readable":im is not None,
                 "width":int(im.shape[1]) if im is not None else None,
                 "height":int(im.shape[0]) if im is not None else None})
    seen.setdefault(h,str(p.relative_to(IMAGE_ROOT)))
inventory=pd.DataFrame(rows)
inventory.to_csv(DIRS["01_Dataset_Inventory"]/"dataset_inventory.csv",index=False)
print("Total:",len(rows),"Readable:",int(inventory.readable.sum()),
      "Unreadable:",int((~inventory.readable).sum()),
      "Duplicate content:",int(inventory.duplicate_of.notna().sum()))
if not inventory.readable.any(): raise RuntimeError("No readable images.")


## 5. Clone official 4D-Humans and install runtime dependencies

The official repository's environment file specifies Python 3.10, PyTorch, CUDA 11.8, torchvision, Lightning, SMPL-X, pyrender, Detectron2 and Chumpy. Kaggle base images vary, so this notebook preserves the installed PyTorch/torchvision pair and tests Detectron2 compilation against the active image. If CUDA toolkit/compiler compatibility fails, it stops rather than guessing a wheel.

In [ ]:
def cmd(args,cwd=None):
    print("$"," ".join(map(str,args)))
    p=subprocess.run(args,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(p.stdout[-10000:])
    if p.returncode: raise RuntimeError("Command failed: "+str(args)+"\n"+p.stdout[-5000:])
if not REPO.exists(): cmd(["git","clone","https://github.com/shubham-goel/4D-Humans.git",str(REPO)])
nvcc=shutil.which("nvcc")
print("nvcc:",nvcc)
if not nvcc: raise RuntimeError("nvcc missing. This Kaggle GPU image cannot compile Detectron2 from source.")
cmd([sys.executable,"-m","pip","install","-U","pip","setuptools","wheel"])
cmd([sys.executable,"-m","pip","install","pytorch-lightning","smplx==0.1.28","pyrender",
     "opencv-python-headless","yacs","scikit-image","einops","timm","dill","pandas",
     "hydra-core","omegaconf","webdataset","gdown","chumpy"])
cmd([sys.executable,"-m","pip","install","-e",".","--no-deps"],cwd=REPO)
os.environ["FORCE_CUDA"]="1"
os.environ["TORCH_CUDA_ARCH_LIST"]="6.0;7.0;7.5;8.0;8.6"
cmd([sys.executable,"-m","pip","install","git+https://github.com/facebookresearch/detectron2.git","--no-build-isolation"])


## 6. Compatibility patches and import verification

Patches are applied in this same kernel process before importing HMR2. The checkpoint loader is explicitly configured for trusted official checkpoint files.

In [ ]:
for alias,value in {"bool":np.bool_,"int":int,"float":float,"complex":complex,"object":object,"str":str}.items():
    if alias not in np.__dict__: setattr(np,alias,value)
import matplotlib
matplotlib.use("Agg",force=True)
_original_load=torch.load
def trusted_load(*args,**kwargs):
    kwargs.setdefault("weights_only",False)
    return _original_load(*args,**kwargs)
torch.load=trusted_load
checks={}
for mod in ["detectron2","pytorch_lightning","smplx","pyrender","chumpy","omegaconf","hmr2"]:
    try: __import__(mod); checks[mod]="OK"
    except Exception as e: checks[mod]=repr(e)
print(json.dumps(checks,indent=2))
bad={k:v for k,v in checks.items() if v!="OK"}
if bad: raise RuntimeError("Dependency import check failed: "+json.dumps(bad,indent=2))
(DIRS["02_Environment_Logs"]/"imports.json").write_text(json.dumps(checks,indent=2))


## 7. Install provided SMPL model and load official HMR2 checkpoint

The supplied neutral model is copied to the path expected by 4D-Humans. It is not downloaded again.

In [ ]:
from hmr2.configs import CACHE_DIR_4DHUMANS
smpl_target=Path(CACHE_DIR_4DHUMANS)/"data"/"smpl"/"SMPL_NEUTRAL.pkl"
smpl_target.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(SMPL,smpl_target)
from hmr2.models import download_models,load_hmr2,DEFAULT_CHECKPOINT
download_models(CACHE_DIR_4DHUMANS)
if not Path(DEFAULT_CHECKPOINT).exists(): raise RuntimeError("Official HMR2 checkpoint is missing after download.")
model,model_cfg=load_hmr2(DEFAULT_CHECKPOINT)
device=torch.device("cuda"); model=model.to(device); model.eval()
print("HMR2 model loaded on",device)


## 8. One-image inference smoke test

This gate must create a rendered PNG and OBJ mesh before full-dataset processing is allowed. The official demo is executed in-process so compatibility patches remain active.

In [ ]:
from hmr2.utils.utils_detectron2 import DefaultPredictor_Lazy
# Run the official demo through runpy with one image.
readable=inventory[inventory.readable]
sample_rel=readable.iloc[0].relative_path
sample=IMAGE_ROOT/sample_rel
smoke=DIRS["03_HMR_Outputs"]/"smoke"; smoke.mkdir(parents=True,exist_ok=True)
sys.argv=["demo.py","--img_folder",str(sample.parent),"--out_folder",str(smoke),
          "--save_mesh","--detector","regnety","--batch_size","1","--file_type",sample.name]
try: runpy.run_path(str(REPO/"demo.py"),run_name="__main__")
except SystemExit as e:
    if e.code not in (None,0): raise
smoke_png=list(smoke.glob("*.png")); smoke_obj=list(smoke.glob("*.obj"))
print("PNG:",len(smoke_png),"OBJ:",len(smoke_obj))
if not smoke_png or not smoke_obj:
    raise RuntimeError("Smoke test did not produce both PNG and OBJ. Do not start full inference.")


## 9. Full-dataset inference with resumable per-image records

One source image is processed per demo invocation for robustness. Each image gets its own output folder. Failed images include full traceback logs. Existing successful PNG+OBJ outputs are skipped on rerun.

In [ ]:
import time
OUT=DIRS["03_HMR_Outputs"]/"per_image"; OUT.mkdir(parents=True,exist_ok=True)
RENDERS=DIRS["04_Rendered_Meshes"]
pred=[]; failures=[]; t0=time.time()
for _,r in inventory.iterrows():
    rel=Path(r.relative_path); src=IMAGE_ROOT/rel; stem=rel.stem; out=OUT/stem; out.mkdir(exist_ok=True)
    if not r.readable:
        failures.append({"image":r.relative_path,"stage":"decode","error":"Unreadable image"}); continue
    oldpng=list(out.glob("*.png")); oldobj=list(out.glob("*.obj"))
    if oldpng and oldobj:
        pred.append({"image":r.relative_path,"status":"resumed","png_count":len(oldpng),"obj_count":len(oldobj)}); continue
    try:
        sys.argv=["demo.py","--img_folder",str(src.parent),"--out_folder",str(out),
                  "--save_mesh","--detector","regnety","--batch_size","1","--file_type",src.name]
        try: runpy.run_path(str(REPO/"demo.py"),run_name="__main__")
        except SystemExit as e:
            if e.code not in (None,0): raise
        png=list(out.glob("*.png")); obj=list(out.glob("*.obj"))
        if not png or not obj: raise RuntimeError("No rendered PNG/OBJ (no person detected or render failed).")
        for p in png: shutil.copy2(p,RENDERS/p.name)
        pred.append({"image":r.relative_path,"status":"success","png_count":len(png),"obj_count":len(obj),
                     "png_files":";".join(p.name for p in png),"obj_files":";".join(p.name for p in obj)})
    except Exception as e:
        tb=traceback.format_exc()
        failures.append({"image":r.relative_path,"stage":"inference","error":str(e),"traceback":tb})
        (DIRS["05_Failed_Images"]/(stem+"_error.txt")).write_text(tb)
    pd.DataFrame(pred).to_csv(DIRS["07_Evaluation"]/"predictions_partial.csv",index=False)
    pd.DataFrame(failures).to_csv(DIRS["05_Failed_Images"]/"failed_partial.csv",index=False)
elapsed=time.time()-t0


## 10. Experiment summary and CSV exports

In [ ]:
pred_df=pd.DataFrame(pred); fail_df=pd.DataFrame(failures)
pred_df.to_csv(DIRS["07_Evaluation"]/"per_image_results.csv",index=False)
fail_df.to_csv(DIRS["05_Failed_Images"]/"failed_images.csv",index=False)
summary={"utc":datetime.now(timezone.utc).isoformat(),"images_total":len(inventory),
 "readable":int(inventory.readable.sum()),"unreadable":int((~inventory.readable).sum()),
 "successful_or_resumed":len(pred_df),"failed":len(fail_df),"runtime_seconds":elapsed,
 "gpu":torch.cuda.get_device_name(0),"torch":torch.__version__,"torch_cuda":torch.version.cuda,
 "repo":"https://github.com/shubham-goel/4D-Humans","asset_source":asset_source,
 "dataset_sha256":sha256(DATA_ZIP),"smpl_sha256":sha256(SMPL),"ground_truth_3d_available":False}
(DIRS["06_Experiment_Metadata"]/"experiment_metadata.json").write_text(json.dumps(summary,indent=2))
(DIRS["07_Evaluation"]/"image_processing_summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
display(pred_df.head(20)); display(fail_df.head(20))


## 11. Visual inspection: up to 10 random predictions

Original and HMR2 output are shown side by side. These are qualitative estimates, not geometric accuracy measurements.

In [ ]:
import matplotlib.pyplot as plt
good=pred_df[pred_df.status.isin(["success","resumed"])] if len(pred_df) else pred_df
sample_rows=good.sample(min(10,len(good)),random_state=42) if len(good) else good
for _,r in sample_rows.iterrows():
    rel=Path(r.image); original=cv2.cvtColor(cv2.imread(str(IMAGE_ROOT/rel)),cv2.COLOR_BGR2RGB)
    outputs=list((OUT/rel.stem).glob("*.png"))
    if not outputs: continue
    rendered=cv2.cvtColor(cv2.imread(str(outputs[0])),cv2.COLOR_BGR2RGB)
    fig,ax=plt.subplots(1,2,figsize=(12,5))
    ax[0].imshow(original); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(rendered); ax[1].set_title("HMR2 prediction"); ax[1].axis("off")
    fig.suptitle(r.image); plt.show()


## 12. Outputs

All persistent artifacts are under `/kaggle/working/Saree_HMR_Research/`. Save a notebook version or use Kaggle's output files to download the folder. OBJ meshes are predictions. Without ground-truth 3D meshes, do not report MPJPE, vertex error, or geometric accuracy.

In [ ]:
for name,folder in DIRS.items():
    count=sum(1 for p in folder.rglob("*") if p.is_file())
    print(name, count, "files:", folder)
print("Complete output root:",ROOT)
